In [12]:
import requests
from time import sleep


url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

all_ids = []
retstart = 0
retmax = 1000

while True:
    params = {
        "db": "protein",
        "term": "uvsx",
        "retstart": retstart,
        "retmax": retmax,
        "retmode": "json"
    }

    res = requests.get(url, params=params).json()
    ids = res["esearchresult"]["idlist"]

    if not ids:
        break

    all_ids.extend(ids)
    retstart += retmax

print(len(all_ids))

1137


In [13]:
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
params = {
    "db": "protein",
    "id": ",".join(all_ids[:200]),  # batch of IDs
    "retmode": "json"
}

res = requests.get(url, params=params, verify=True)
data = res.json()

# extract accessions
accessions = []

for uid, rec in data["result"].items():
    if uid == "uids":
        continue  # skip the list of IDs
    accessions.append(rec.get("caption"))

print(accessions)

['P32270', 'YDI93881', 'YDI88709', 'YDD38469', '9GBG_A', 'YCR04191', 'YCQ78089', 'YCQ76618', 'P04529', 'O21960', 'O21959', 'O21958', 'O21957', 'O21956', 'O21955', 'O21954', 'O21953', 'O21952', 'O21951', 'O21950', 'O21949', 'O21947', 'O21948', 'Q06728', 'Q06727', 'P04537', 'P03690', 'YCL39723', 'YCJ26573', 'YCJ26308', 'WIL02624', 'WIL02365', 'QSM00476', 'YCB26705', 'YBW68348', 'YBW02273', 'YBB95625', 'UCR81688', 'WNM70829', 'YP_013606416', 'XEN39446', 'XYL45948', 'XYL45353', 'XXS07581', 'XXL68479', 'XXJ63011', 'XXJ62871', 'XTB82809', 'XAN60209', 'XQU42776', 'XQU59788', 'XQU49485', 'XQO94789', 'XPO95924', 'XOS61644', 'XOS51285', 'XOS25815', 'XOS24228', 'XOR95616', 'XOR82736', 'XOR76507', 'XOR74364', 'XOR59935', 'XOR59488', 'XOR35443', 'XOR35128', 'XOR34618', 'XOR33923', 'XOR33708', 'XOR17395', 'XOR06429', 'XOR06191', 'XOR06060', 'XOQ89880', 'XOD32333', 'XOD32076', 'XOD31821', 'XOD31540', 'XOA07903', 'XLQ30668', 'XMR90074', 'XDR01266', 'XLQ29474', 'XLM56194', 'XBA89584', 'XLL18380', 'XAG9

In [14]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
batch_size = 200
all_sequences = ""

for start in range(0, len(all_ids), batch_size):
    batch = all_ids[start:start + batch_size]
    params = {
        "db": "protein",
        "id": ",".join(batch),
        "rettype": "fasta",
        "retmode": "text"
    }
    res = requests.get(fetch_url, params=params)
    all_sequences += res.text
    print(f"Fetched sequences {start + 1} to {start + len(batch)}")
    sleep(0.4)  # polite delay to avoid hitting NCBI limits

    with open("../data/ncbi/uvsx_sequences.fasta", "w") as f:
        f.write(all_sequences)

print("All sequences saved to 'uvsx_sequences.fasta'")

Fetched sequences 1 to 200
Fetched sequences 201 to 400
Fetched sequences 401 to 600
Fetched sequences 601 to 800
Fetched sequences 801 to 1000
Fetched sequences 1001 to 1137
All sequences saved to 'uvsx_sequences.fasta'


In [69]:
import requests
import time

batch_size = 200
summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
metadata_records = []

for start in range(0, len(all_ids), batch_size):
    batch = all_ids[start:start + batch_size]
    params = {
        "db": "protein",
        "id": ",".join(batch),
        "retmode": "json"
    }
    res = requests.get(summary_url, params=params).json()

    # Process this batch immediately
    for uid, rec in res["result"].items():
        if uid == "uids":  # skip the uids list
            continue

        subtype = rec.get("subtype")
        subname = rec.get("subname")

        # Base record
        record = {
            "uid": rec.get("uid"),
            "accession": rec.get("caption"),
            "title": rec.get("title"),
            "organism": rec.get("organism"),
            "sequence_length": rec.get("slen"),
            "create_date": rec.get("createdate"),
            "update_date": rec.get("updatedate"),
            "dbsource": rec.get("sourcedb"),
            "extra": rec.get("extra")
        }

        # Add subtype/subname if present
        if subtype and subname:
            type_parts = subtype.split('|')
            name_parts = subname.split('|')
            for t, n in zip(type_parts, name_parts):
                if t and n:
                    record[t] = n

        metadata_records.append(record)

    print(f"Processed batch {start+1} to {start+len(batch)}")
    time.sleep(0.4)

print(f"Total records collected: {len(metadata_records)}")

Processed batch 1 to 200
Processed batch 201 to 400
Processed batch 401 to 600
Processed batch 601 to 800
Processed batch 801 to 1000
Processed batch 1001 to 1137
Total records collected: 1137


In [79]:
with open("../data/ncbi/uvsx_metadata.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames=[]
    for i in metadata_records:
        fields=i.keys()
        for f in fields:
            if f not in fieldnames:
                fieldnames.append(f)
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(metadata_records)

print("Metadata saved to 'uvsx_metadata.csv'")

Metadata saved to 'uvsx_metadata.csv'
